
# 🌱 S6E4 Kaggle Notebook (Stabilan + Edukativan)

Cilj: da notebook **uvek prođe do kraja** i da bude pregledniji za Kaggle.

Šta smo unapredili:
- robust učitavanje paketa (XGBoost fallback ako ne radi),
- jasan EDA sa lepšim grafikonima,
- feature engineering sa komentarima,
- CV treniranje više modela + blend,
- threshold tuning i generisanje `submission.csv`.


In [ ]:

import os
import gc
import random
import warnings
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import HistGradientBoostingClassifier
from scipy.optimize import minimize

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
print('Setup je spreman ✅')


In [ ]:

# Optional installs (Kaggle obično već ima ove biblioteke)
try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:
    HAS_LGB = False

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
except Exception:
    HAS_CAT = False

print({'lightgbm': HAS_LGB, 'xgboost': HAS_XGB, 'catboost': HAS_CAT})
if not HAS_XGB:
    print('ℹ️ XGBoost nije dostupan -> koristićemo sklearn fallback, pa notebook i dalje prolazi.')


In [ ]:

DATA_DIRS = ['/kaggle/input/playground-series-s6e4', '/kaggle/input', '/content', '.']

def find_file(filename: str) -> str:
    for d in DATA_DIRS:
        p = os.path.join(d, filename)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'{filename} not found')

train = pd.read_csv(find_file('train.csv'))
test = pd.read_csv(find_file('test.csv'))
sample = pd.read_csv(find_file('sample_submission.csv'))

TARGET = 'Irrigation_Need'
ID_COL = 'id'
LABELS = ['Low', 'Medium', 'High']
LABEL_TO_INT = {k: i for i, k in enumerate(LABELS)}
INT_TO_LABEL = {i: k for k, i in LABEL_TO_INT.items()}

NUM_COLS = ['Soil_Moisture', 'Rainfall_mm', 'Temperature_C', 'Wind_Speed_kmh', 'Humidity']
CAT_COLS = [c for c in train.columns if c not in NUM_COLS + [ID_COL, TARGET]]

print('train:', train.shape, '| test:', test.shape)
display(train.head(3))


## 📊 Brzi EDA (vizuelno lepše + korisno za odluke modela)

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=train, x=TARGET, order=LABELS, palette='viridis', ax=axes[0])
axes[0].set_title('Distribucija target klase')
axes[0].set_xlabel('Klasa')
axes[0].set_ylabel('Broj redova')

corr = train[NUM_COLS].corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1])
axes[1].set_title('Korelacija numeričkih feature-a')

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(NUM_COLS):
    sns.histplot(train[col], kde=True, ax=axes[i], color='#2c7fb8')
    axes[i].set_title(f'Distribucija: {col}')
axes[-1].axis('off')
plt.tight_layout()
plt.show()


In [ ]:

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Domain-inspired scoring
    df['soil_lt_25'] = (df['Soil_Moisture'] < 25).astype(int)
    df['rain_lt_300'] = (df['Rainfall_mm'] < 300).astype(int)
    df['temp_gt_30'] = (df['Temperature_C'] > 30).astype(int)
    df['wind_gt_10'] = (df['Wind_Speed_kmh'] > 10).astype(int)

    df['high_score'] = 2 * df['soil_lt_25'] + 2 * df['rain_lt_300'] + df['temp_gt_30'] + df['wind_gt_10']
    df['is_harvest'] = (df['Crop_Growth_Stage'] == 'Harvest').astype(int)
    df['is_sowing'] = (df['Crop_Growth_Stage'] == 'Sowing').astype(int)
    df['mulch_yes'] = (df['Mulching_Used'] == 'Yes').astype(int)
    df['low_score'] = 2 * df['is_harvest'] + 2 * df['is_sowing'] + df['mulch_yes']
    df['magic_score'] = df['high_score'] - df['low_score']
    df['formula_pred_int'] = np.where(df['magic_score'] <= 0, 0, np.where(df['magic_score'] <= 3, 1, 2))

    # Interaction features
    df['heat_index'] = df['Temperature_C'] * (df['Humidity'] / 100.0)
    df['moisture_stress'] = df['Soil_Moisture'] / (df['Temperature_C'] + 1e-6)
    df['rain_per_temp'] = df['Rainfall_mm'] / (df['Temperature_C'] + 1e-6)
    df['dry_and_hot'] = ((df['Soil_Moisture'] < 20) & (df['Temperature_C'] > 35)).astype(int)

    for col in ['Soil_Type', 'Region']:
        grp = df.groupby(col)['Soil_Moisture'].transform('mean')
        df[f'{col}_mean_moisture'] = grp
        df[f'{col}_moisture_diff'] = df['Soil_Moisture'] - grp

    return df


def encode_categories(train_df: pd.DataFrame, test_df: pd.DataFrame, cat_cols: List[str]):
    train_df = train_df.copy()
    test_df = test_df.copy()
    for col in cat_cols:
        cats = pd.Index(pd.concat([train_df[col].astype(str), test_df[col].astype(str)], axis=0).unique())
        train_df[col] = pd.Categorical(train_df[col].astype(str), categories=cats).codes.astype(np.int32)
        test_df[col] = pd.Categorical(test_df[col].astype(str), categories=cats).codes.astype(np.int32)
    return train_df, test_df


train_fe = add_features(train)
test_fe = add_features(test)
train_fe, test_fe = encode_categories(train_fe, test_fe, CAT_COLS)

feature_cols = [c for c in train_fe.columns if c not in [ID_COL, TARGET]]
X = train_fe[feature_cols].copy()
X_test = test_fe[feature_cols].copy()
y = train_fe[TARGET].map(LABEL_TO_INT).astype(int).values

print('Broj feature-a:', len(feature_cols))
print('X shape:', X.shape, '| X_test shape:', X_test.shape)


In [ ]:

def tune_thresholds(y_true: np.ndarray, probs: np.ndarray) -> np.ndarray:
    def objective(th):
        return -balanced_accuracy_score(y_true, (probs * th).argmax(axis=1))
    res = minimize(objective, x0=np.ones(probs.shape[1]), method='Nelder-Mead', tol=1e-6)
    return res.x


def cv_model(name: str, X: pd.DataFrame, y: np.ndarray, X_test: pd.DataFrame, n_splits: int = 5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof = np.zeros((len(X), 3), dtype=np.float32)
    tst = np.zeros((len(X_test), 3), dtype=np.float32)
    scores = []

    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        sw = compute_sample_weight(class_weight='balanced', y=y[tr])

        if name == 'lgbm':
            model = lgb.LGBMClassifier(
                objective='multiclass', num_class=3, n_estimators=8000,
                learning_rate=0.025, num_leaves=128, subsample=0.85,
                colsample_bytree=0.85, random_state=SEED + fold, n_jobs=-1, verbose=-1
            )
            model.fit(X.iloc[tr], y[tr], sample_weight=sw,
                      eval_set=[(X.iloc[va], y[va])], eval_metric='multi_logloss',
                      callbacks=[lgb.early_stopping(200, verbose=False)])

        elif name == 'xgb':
            model = xgb.XGBClassifier(
                objective='multi:softprob', num_class=3, n_estimators=7000,
                learning_rate=0.03, max_depth=7, subsample=0.85,
                colsample_bytree=0.85, reg_lambda=1.2, tree_method='hist',
                eval_metric='mlogloss', random_state=SEED + fold, n_jobs=-1
            )
            model.fit(X.iloc[tr], y[tr], sample_weight=sw, eval_set=[(X.iloc[va], y[va])], verbose=False)

        elif name == 'cat':
            model = CatBoostClassifier(
                loss_function='MultiClass', eval_metric='MultiClass',
                iterations=7000, learning_rate=0.03, depth=8,
                random_seed=SEED + fold, auto_class_weights='Balanced',
                od_type='Iter', od_wait=200, verbose=0
            )
            cat_idx = [X.columns.get_loc(c) for c in CAT_COLS if c in X.columns]
            model.fit(X.iloc[tr], y[tr], cat_features=cat_idx, eval_set=(X.iloc[va], y[va]), use_best_model=True, verbose=0)

        else:  # fallback kada XGB nije dostupan
            model = HistGradientBoostingClassifier(
                learning_rate=0.05, max_depth=8, max_iter=500,
                random_state=SEED + fold
            )
            model.fit(X.iloc[tr], y[tr], sample_weight=sw)

        pv = model.predict_proba(X.iloc[va])
        pt = model.predict_proba(X_test)
        oof[va] = pv
        tst += pt / n_splits
        sc = balanced_accuracy_score(y[va], pv.argmax(axis=1))
        scores.append(sc)
        print(f'{name.upper()} fold {fold}: {sc:.6f}')

        del model, pv, pt
        gc.collect()

    return oof, tst, scores


In [ ]:

results: Dict[str, Dict[str, np.ndarray]] = {}

models_to_run = []
if HAS_LGB:
    models_to_run.append('lgbm')
if HAS_XGB:
    models_to_run.append('xgb')
else:
    models_to_run.append('hgb')
if HAS_CAT:
    models_to_run.append('cat')

print('Modeli koji se treniraju:', models_to_run)

for m in models_to_run:
    oof, tst, scores = cv_model(m, X, y, X_test, n_splits=5)
    results[m] = {'oof': oof, 'tst': tst, 'scores': np.array(scores)}

score_table = []
for m, r in results.items():
    sc = balanced_accuracy_score(y, r['oof'].argmax(axis=1))
    score_table.append((m, sc))

score_df = pd.DataFrame(score_table, columns=['model', 'oof_bal_acc']).sort_values('oof_bal_acc', ascending=False)
display(score_df)


In [ ]:

# Weighted blend po OOF performansi
model_names = score_df['model'].tolist()
weights = np.array([1.0 / max(score_df.loc[score_df['model'] == m, 'oof_bal_acc'].values[0], 1e-6) for m in model_names])
weights = weights / weights.sum()

print('Blend modeli:', model_names)
print('Blend težine:', weights)

oof_blend = np.zeros((len(X), 3), dtype=np.float32)
tst_blend = np.zeros((len(X_test), 3), dtype=np.float32)
for w, m in zip(weights, model_names):
    oof_blend += w * results[m]['oof']
    tst_blend += w * results[m]['tst']

print('OOF pre threshold tuning:', balanced_accuracy_score(y, oof_blend.argmax(axis=1)))

thresholds = tune_thresholds(y, oof_blend)
print('Optimizovani threshold-i:', thresholds)

final_test_pred = (tst_blend * thresholds).argmax(axis=1)

# Pravilo za ekstremne slučajeve (domain heuristic)
extreme_mask = (test_fe['magic_score'].values <= -2) | (test_fe['magic_score'].values >= 5)
final_test_pred = np.where(extreme_mask, test_fe['formula_pred_int'].values, final_test_pred)

submission = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    TARGET: [INT_TO_LABEL[int(x)] for x in final_test_pred]
})
submission.to_csv('submission.csv', index=False)

print('submission.csv sačuvan ✅')
display(submission.head())


In [ ]:

# Vizuelna provera distribucije train vs submission
train_dist = train[TARGET].value_counts(normalize=True).rename('train_share')
sub_dist = submission[TARGET].value_counts(normalize=True).rename('submission_share')
comp = pd.concat([train_dist, sub_dist], axis=1).fillna(0).reindex(LABELS)
comp['diff'] = comp['submission_share'] - comp['train_share']
display(comp)

ax = comp[['train_share', 'submission_share']].plot(kind='bar', figsize=(10, 5), colormap='viridis')
ax.set_title('Train vs Submission distribucija')
ax.set_ylabel('Udeo')
plt.xticks(rotation=0)
plt.show()
